# 🔧 Tools & Functions Essentials

## Learning Objectives
In this notebook, you will learn:
1. **What Tools Are** - Understanding LangChain tools and their role in extending LLM capabilities
2. **External API Integration** - Building custom tools that fetch real-world data from APIs
3. **ReAct Agents** - Creating agents that reason and act using tools
4. **Multi-Tool Agents** - Combining multiple tools into a single agent workflow
5. **Multi-Step Chains with LCEL** - Building parallel execution pipelines using LangChain Expression Language

## Prerequisites
- Basic understanding of LangChain and LLMs
- A `.env` file with `WEATHER_API_KEY` configured (OpenWeatherMap)
- Packages from `pyproject.toml` installed

---

## 📖 What Are Tools in LangChain?

Tools in LangChain are external functionalities that an LLM can leverage during its workflow. These functionalities are defined as actions or tasks that extend the model's capabilities beyond simple text generation.

### Key Concepts:
- **Dynamic Integration**: Tools enable LLMs to interact with real-world systems, such as APIs or databases, in real-time
- **Modularity**: You can add or remove tools easily, creating flexible and reusable workflows
- **Extensibility**: Custom tools can be defined to handle specific use cases, such as fetching data from a proprietary system or performing domain-specific calculations

### Common Use Cases
- **API Integration**: Calling weather APIs, stock market data, or translation services
- **Database Queries**: Retrieving records from SQL or NoSQL databases
- **Custom Scripts**: Executing Python functions or scripts to process data dynamically

In [22]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and Configuration
# ============================================================================

import os
import sys
import warnings

warnings.filterwarnings("ignore")

sys.path.append(os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))

print("✅ Environment configured successfully!")

✅ Environment configured successfully!


In [23]:
# ============================================================================
# LLM INITIALIZATION: Load Language Model
# ============================================================================

from helpers.utils import get_llm

llm = get_llm()
# llm = get_llm("openai")
# llm = get_llm("groq")

LLM initialized: databricks-claude-opus-4-6 (via databricks)


---

## 🌐 Part 1: Integrating an External API as a Tool

In this section, we'll build a custom tool that fetches real-time weather data from the OpenWeatherMap API. This demonstrates how to wrap any Python function as a LangChain tool that an agent can use.

In [24]:
# ============================================================================
# WEATHER FUNCTION: Fetch Real-Time Weather Data
# ============================================================================

import requests

def fetch_weather(location):
    """Fetch current weather data for a given location using OpenWeatherMap API."""
    api_key = os.getenv("WEATHER_API_KEY")
    url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&appid={api_key}"
    try:
        response = requests.get(url)
        data = response.json()
        if response.status_code == 200:
            weather = data["weather"][0]["description"]
            temperature = data["main"]["temp"]
            return f"The weather in {location} is {weather} with a temperature of {temperature} kelvin."
        else:
            return f"❌ Error: {data['message']}"
    except Exception as e:
        return f"❌ Error: {e}"

print("✅ Weather function defined!")

✅ Weather function defined!


In [25]:
# ============================================================================
# TEST: Verify Weather Function
# ============================================================================

result = fetch_weather("kolkata")
print(f"🔍 {result}")

🔍 The weather in kolkata is broken clouds with a temperature of 296.74 kelvin.


### 🔧 Wrapping the Function as a LangChain Tool

To make our function usable by an agent, we wrap it in a `Tool` object. This provides the agent with a name, description, and callable function it can invoke during reasoning.

In [26]:
# ============================================================================
# WEATHER TOOL: Wrap Function as LangChain Tool
# ============================================================================

from langchain_core.tools import Tool

weather_tool = Tool(
    name="WeatherTool",
    func=fetch_weather,
    description="A tool for fetching current weather data of a location"
)

print(f"✅ Tool created: {weather_tool.name}")
print(f"📋 Description: {weather_tool.description}")

✅ Tool created: WeatherTool
📋 Description: A tool for fetching current weather data of a location


In [36]:
weather_tool.invoke("Kolkata")

'The weather in Kolkata is broken clouds with a temperature of 296.74 kelvin.'

### 🤖 Creating a ReAct Agent

A ReAct (Reasoning + Acting) agent uses an LLM to decide **when** and **how** to use tools. The agent receives a user query, reasons about what tools to call, executes them, and synthesizes a final response.

In [27]:
# ============================================================================
# REACT AGENT: Create Agent with Weather Tool
# ============================================================================

from langgraph.prebuilt import create_react_agent

weather_agent = create_react_agent(llm, tools=[weather_tool])

print("✅ Weather agent created!")

✅ Weather agent created!


In [28]:
# ============================================================================
# TEST: Query the Weather Agent
# ============================================================================

query = "What is the weather in Kolkata?"
response = weather_agent.invoke({"messages": [("user", query)]})

print(f"🤖 Agent Response:\n{response['messages'][-1].content}")

🤖 Agent Response:
Here's the current weather in **Kolkata**:

- **Condition:** Broken clouds ☁️
- **Temperature:** ~23.6°C (74.5°F)

It's a relatively mild and cloudy day in Kolkata! Let me know if you'd like any more details.


---

## 🕐 Part 2: Building a Multi-Tool Agent

Now we'll extend our agent by adding a second tool — a timezone lookup. This demonstrates how agents can orchestrate multiple tools to answer complex questions that require information from different sources.

### Exercise:
- Create a function for fetching timezone data
- Wrap it as a `Tool` object
- Add both tools to a single agent and test the combined workflow

In [29]:
# ============================================================================
# TIMEZONE FUNCTION: Fetch Timezone Data
# ============================================================================

def fetch_time_zone(location):
    """Fetch timezone information for a given location."""
    try:
        time_zones = {
            "Kolkata": "Asia/Kolkata",
            "New York": "America/New_York",
            "London": "Europe/London",
            "Tokyo": "Asia/Tokyo",
            "Sydney": "Australia/Sydney",
            "Paris": "Europe/Paris",
        }
        tz = time_zones.get(location, "Unknown")
        return f"The time zone of {location} is {tz}"
    except Exception as e:
        return f"❌ Error: {e}"

print("✅ Timezone function defined!")

✅ Timezone function defined!


In [30]:
# ============================================================================
# MULTI-TOOL AGENT: Combine Weather + Timezone Tools
# ============================================================================

time_zone_tool = Tool(
    name="TimeZoneTool",
    func=fetch_time_zone,
    description="A tool for fetching timezone data of a location"
)

agent = create_react_agent(llm, tools=[weather_tool, time_zone_tool])

print(f"✅ Tools: {[t.name for t in [weather_tool, time_zone_tool]]}")
print("✅ Multi-tool agent created!")

✅ Tools: ['WeatherTool', 'TimeZoneTool']
✅ Multi-tool agent created!


In [31]:
# ============================================================================
# TEST: Query the Multi-Tool Agent
# ============================================================================

query = "Tell me the weather and timezone of Kolkata"
response = agent.invoke({"messages": [("user", query)]})

print(f"🤖 Agent Response:\n{response['messages'][-1].content}")

🤖 Agent Response:
Here's the information for **Kolkata**:

### 🌤️ Weather
- **Condition:** Broken clouds
- **Temperature:** ~23.6°C (296.74 K / ~74.5°F)

### 🕐 Timezone
- **Timezone:** Asia/Kolkata (IST - Indian Standard Time, UTC +5:30)

It looks like Kolkata is experiencing partly cloudy skies with a pleasant temperature. Let me know if you need any more details!


---

## ⛓️ Part 3: Multi-Step Chains with LCEL

LangChain Expression Language (LCEL) allows you to compose chains using the pipe (`|`) operator. In this section, we build a workflow that:

1. Accepts a location as input
2. Runs weather and timezone lookups **in parallel** using `RunnableParallel`
3. Feeds both results into a summary chain

> **Note**: Unlike the agent approach above (which uses tool-calling), this approach uses prompt chains — each step is an LLM call with a specialized prompt. The LLM generates the information from its knowledge rather than calling external APIs.

In [32]:
# ============================================================================
# PROMPTS: Define Templates for Each Chain Step
# ============================================================================

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

weather_prompt = PromptTemplate(
    input_variables=["location"],
    template="What is the weather in {location}?"
)

time_zone_prompt = PromptTemplate(
    input_variables=["location"],
    template="Fetch the timezone for {location}?"
)

summary_prompt = PromptTemplate(
    input_variables=["weather_info", "time_zone_info"],
    template="Provide a Summary: {weather_info} and {time_zone_info}"
)

print("✅ Prompt templates defined!")

✅ Prompt templates defined!


In [41]:
# ============================================================================
# LCEL CHAINS: Build Parallel Execution Pipeline
# ============================================================================

from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

# Use the actual API tools (via RunnableLambda) for real-time data
weather_chain = RunnableLambda(lambda x: fetch_weather(x["location"]))
time_zone_chain = RunnableLambda(lambda x: fetch_time_zone(x["location"]))

# Summary chain uses the LLM to synthesize the tool results
summary_chain = summary_prompt | llm | parser

# --- Compose the full pipeline ---
# RunnableParallel runs weather & timezone lookups concurrently,
# then feeds both results into the summary chain
multi_step_chain = (
    RunnableParallel(
        weather_info=weather_chain,
        time_zone_info=time_zone_chain,
    )
    | summary_chain
)

print("✅ Multi-step chain assembled (using real API tools)!")

✅ Multi-step chain assembled (using real API tools)!


In [42]:
# ============================================================================
# TEST: Run the Multi-Step Chain
# ============================================================================

response = multi_step_chain.invoke({"location": "Kolkata"})

print(f"🤖 Chain Response:\n{response}")

🤖 Chain Response:
## Summary

**Kolkata Weather & Time Zone Overview**

- **Current Weather:** The sky in Kolkata is covered with **overcast clouds**.
- **Temperature:** The current temperature is **297.37 K**, which is approximately **24.22°C (75.6°F)**, indicating warm and humid conditions.
- **Time Zone:** Kolkata operates under the **Asia/Kolkata** time zone, which is **UTC +5:30** (Indian Standard Time).

Overall, Kolkata is experiencing a typical warm and cloudy day, consistent with its tropical wet-and-dry climate.


In [ ]:
# ============================================================================
# TEST: Run an Individual Chain (uses real API)
# ============================================================================

individual_response = weather_chain.invoke({"location": "Kolkata"})

print(f"🔍 Individual Weather Chain:\n{individual_response}")

---

## 📝 Summary

In this notebook, we learned:

### 1. Tools in LangChain
- **Tools** extend LLM capabilities by connecting them to external functions, APIs, and data sources
- Any Python function can be wrapped as a `Tool` with a name and description

### 2. ReAct Agents
- **ReAct agents** use reasoning to decide which tools to call and when
- Created with `create_react_agent()` from LangGraph
- Agents can orchestrate **multiple tools** in a single conversation

### 3. Multi-Step Chains with LCEL
- **LCEL** (LangChain Expression Language) uses the pipe operator (`|`) to compose chains
- **`RunnableParallel`** runs multiple chains concurrently for efficiency
- Chains can be composed, tested individually, and combined into complex workflows

### Next Steps
- Explore **Notebook 6.1**: Tool Calling with LangChain — learn about the `@tool` decorator and structured tool calling
- Try adding more tools (e.g., a currency converter, news API)
- Experiment with error handling and tool fallbacks